# 5주차 — SELECT ③ 계산·CASE·집계
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다.
어떤 셀이 오류로 막히거나 데이터가 엉켰다면, 아무 셀에서 `reset_db()`를 실행하면 처음 상태로 돌아갑니다.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, shutil, sqlite3, urllib.request
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/data-v1/mini_emr/mini_emr.sqlite"
DB_PATH = "mini_emr.sqlite"
WORK_PATH = DB_PATH   # 실습에 사용할 파일

con = None

def reset_db():
    """DB를 원본 상태로 되돌리고 다시 연결 (막히면 언제든 호출)"""
    global con
    if con is not None:
        con.close()
    try:
        urllib.request.urlretrieve(DB_URL, DB_PATH)
    except Exception as e:
        if not os.path.exists(DB_PATH):
            raise RuntimeError(
                "DB 다운로드 실패 — 왼쪽 파일 탭에 mini_emr.sqlite를 업로드한 뒤 이 셀을 다시 실행하세요."
            ) from e
        print("다운로드 실패 — 업로드되어 있는 파일을 사용합니다.")
    if WORK_PATH != DB_PATH:
        shutil.copyfile(DB_PATH, WORK_PATH)   # 원본 보존, 실습은 사본에서
    con = sqlite3.connect(WORK_PATH)
    con.execute("PRAGMA foreign_keys = ON")   # SQLite는 기본으로 꺼져 있음 — 반드시 켠다
    assert con.execute("PRAGMA foreign_keys").fetchone()[0] == 1

def q(sql):
    """조회(SELECT) 실행 → 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """데이터/구조를 바꾸는 SQL(INSERT/UPDATE/DELETE/CREATE 등) 실행.
    여러 문장을 순서대로 실행하고 즉시 커밋합니다 — 트랜잭션(BEGIN/ROLLBACK) 실습은
    8주차처럼 con.cursor()를 직접 사용하세요."""
    con.executescript(sql); con.commit(); print("실행 완료")

reset_db()
q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 1. 산술식 — 재원일수

In [ ]:
# 입원별 재원일수(LOS)
q("""
SELECT visit_id, visit_date, discharge_date,
       julianday(discharge_date) - julianday(visit_date) AS los
FROM visit WHERE visit_type = '입원' ORDER BY los DESC LIMIT 8
""")

In [ ]:
# 나이 계산 — substr(연도 자르기) + CAST(숫자 변환), 연 나이 근사
q("""
SELECT patient_id, birth_date,
       2026 - CAST(substr(birth_date,1,4) AS INT) AS age
FROM patient ORDER BY age DESC LIMIT 5
""")

## 2. CASE — 측정치 구간 분류

In [ ]:
# 혈압 측정치 3구간 (측정 1회의 구간 분류 — 진단 아님, 경계값은 예시)
q("""
SELECT visit_id, sbp, dbp,
  CASE WHEN sbp >= 140 OR dbp >= 90 THEN '고혈압 범위'
       WHEN sbp >= 120 THEN '주의 범위'
       ELSE '정상 범위' END AS bp_range
FROM vital ORDER BY sbp DESC LIMIT 10
""")

## 3. 집계함수

In [ ]:
# 입원 요약: 건수·평균·최장
q("""
SELECT COUNT(*) AS 입원건수,
       ROUND(AVG(julianday(discharge_date) - julianday(visit_date)), 1) AS 평균재원일,
       MAX(julianday(discharge_date) - julianday(visit_date)) AS 최장재원일
FROM visit WHERE visit_type = '입원'
""")

In [ ]:
# 정상범위 벗어난 검사 비율 (CASE×SUM 패턴) — 분모는 결과값이 있는 검사만
q("""
SELECT COUNT(*) AS 전체검사,
  COUNT(result_value) AS 판정가능검사,
  SUM(CASE WHEN result_value > ref_high OR result_value < ref_low THEN 1 ELSE 0 END) AS 이상건수,
  ROUND(100.0 * SUM(CASE WHEN result_value > ref_high OR result_value < ref_low THEN 1 ELSE 0 END) / COUNT(result_value), 1) AS 이상비율
FROM lab
""")

분모를 `COUNT(*)`로 두면 결과값이 NULL이라 판정 불가능한 검사까지 '정상'처럼 분모에 들어갑니다. 이 데이터셋의 정상범위(ref_low/ref_high)는 모두 채워져 있어 `COUNT(result_value)`면 충분합니다.

In [ ]:
# COUNT(*) vs COUNT(컬럼) — NULL의 조용한 제외
q("""
SELECT COUNT(*) AS 전체행수, COUNT(result_value) AS 값있는행수,
       COUNT(*) - COUNT(result_value) AS 결측행수
FROM lab
""")

## 연습문제

In [ ]:
# [연습 Q1] 처방의 평균 투약일수는?
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] 맥박을 서맥(<60)·정상·빈맥(>100)으로 분류해 보라
# 힌트: CASE
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] HbA1c 검사의 최솟값·최댓값·평균은?
# 힌트: WHERE test_name='HbA1c'
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q4] HbA1c가 7.0 이상인 검사 건수의 비율은?
# 힌트: CASE×SUM, 100.0
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q5] 입원 재원일수가 14일을 넘는 내원은 몇 건인가?
q("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. 평균 투약일수**
```sql
SELECT ROUND(AVG(duration_days),1) FROM prescription;
```

**A2. 맥박 분류**
```sql
SELECT visit_id, pulse, CASE WHEN pulse < 60 THEN '서맥' WHEN pulse > 100 THEN '빈맥' ELSE '정상' END AS grade FROM vital;
```

**A3. HbA1c 요약**
```sql
SELECT MIN(result_value), MAX(result_value), ROUND(AVG(result_value),1) FROM lab WHERE test_name='HbA1c';
```

**A4. HbA1c≥7.0 비율**
```sql
SELECT ROUND(100.0*SUM(CASE WHEN result_value>=7.0 THEN 1 ELSE 0 END)/COUNT(*),1) FROM lab WHERE test_name='HbA1c';
```

**A5. 14일 초과 입원**
```sql
SELECT COUNT(*) FROM visit WHERE visit_type='입원' AND julianday(discharge_date)-julianday(visit_date) > 14;
```